# Appendix C: Capstone Project: Real-World Credit Risk Engine
### *Modern Machine Learning & Data Science Engineering* by Vaibhav Rastogi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/blob/main/capstone/credit_risk_engine.ipynb)

---

## 📌 Notebook Overview
The complete end-to-end production credit risk engine: Data ingestion, Scikit-Learn ColumnTransformer pipeline, model training, threshold tuning, and simulated live inference.

In [ ]:
# Environment & Data Auto-Download for Google Colab
import os
import urllib.request

os.makedirs('../data', exist_ok=True)
data_path = "../data/credit_risk_dataset.csv"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/v893979/Modern_Machine_Learning_and_Data_Science_Engineering/main/data/credit_risk_dataset.csv"
    print(f"Downloading credit_risk_dataset.csv from GitHub...")
    urllib.request.urlretrieve(url, data_path)
    print("Download complete!")


### 1. Data Ingestion & Pipeline Definition

Build a leak-free ColumnTransformer with StandardScaler and OneHotEncoder:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve

df = pd.read_csv("../data/credit_risk_dataset.csv")
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

num_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols)
    ]
)

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
model_pipeline.fit(X_train, y_train)

auc = roc_auc_score(y_test, model_pipeline.predict_proba(X_test)[:, 1])
print(f"Out-of-Sample ROC-AUC Score: {auc:.4f}")

### 2. Live Customer Application Scoring

Score a live applicant and make an automated approval/rejection decision:

In [ ]:
new_applicant = pd.DataFrame([{
    "person_age": 28,
    "person_income": 82000,
    "person_home_ownership": "MORTGAGE",
    "person_emp_length": 4,
    "loan_intent": "HOMEIMPROVEMENT",
    "loan_grade": "B",
    "loan_amnt": 18000,
    "loan_int_rate": 9.25,
    "loan_percent_income": 0.22,
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 6
}])

prob = model_pipeline.predict_proba(new_applicant)[0, 1]
decision = "DECLINE / MANUAL REVIEW" if prob >= 0.35 else "AUTOMATIC APPROVAL"
print(f"Predicted Probability of Default: {prob:.2%}")
print(f"Underwriting Decision: {decision}")